## Create Indexes For Github Data

In [ ]:
!pip install -q databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
# --------------------------------------------
# 1. Dummy Repo (Simulating GitHub Structure) -- > Will be repalced with github repo APIs
# 1.1 This can be setup as a Github action or batch  -- > As soon as some code is commited , it triggers the sync.


# Unity Catalog configuration
CATALOG = "main"
SCHEMA = "sandesk4"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.git_index_raw"

# --------------------------------------------
# 1.2 Fetch repo from GitHub API (same format as get_dummy_repo_data)
import requests
import base64
import os

def _load_github_token():
    """Load GITHUB_ACCESS_TOKEN from .env or environment."""
    token = os.environ.get("GITHUB_ACCESS_TOKEN")
    if token:
        return token
    try:
        from dotenv import load_dotenv
        from pathlib import Path
        # Search for .env in cwd and parent dirs (notebook may run from Code_Agent/ or project root)
        current = Path.cwd()
        for _ in range(6):
            env_path = current / ".env"
            if env_path.exists():
                load_dotenv(env_path)
                return os.environ.get("GITHUB_ACCESS_TOKEN")
            if current == current.parent:
                break
            current = current.parent
        load_dotenv()  # fallback: default search
    except ImportError:
        pass
    return os.environ.get("GITHUB_ACCESS_TOKEN")

def get_repo_data_from_github(repo_url, token=None, branch="main"):
    """
    Fetch repo contents from GitHub API. Returns same format as get_dummy_repo_data():
    { "folder_name": { "filename": "file_content", ... }, ... }
    
    Args:
        repo_url: e.g. "https://github.com/Sourav692/repo_dummy" or "Sourav692/repo_dummy"
        token: GitHub token (default: from GITHUB_ACCESS_TOKEN in .env)
        branch: Branch to fetch (default: main)
    """
    token = token or _load_github_token()
    if not token:
        raise ValueError("GITHUB_ACCESS_TOKEN not found. Set it in .env or pass token=...")
    
    # Parse owner/repo from URL
    if "github.com" in repo_url:
        parts = repo_url.rstrip("/").split("github.com/")[-1].split("/")
        owner, repo = parts[0], parts[1]
    else:
        owner, repo = repo_url.split("/")
    
    headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}
    api_base = f"https://api.github.com/repos/{owner}/{repo}/contents"
    
    def fetch_dir(path=""):
        """Recursively fetch directory contents. Returns {folder: {filename: content}}."""
        result = {}
        url = f"{api_base}/{path}" if path else api_base
        resp = requests.get(url, headers=headers, params={"ref": branch})
        resp.raise_for_status()
        
        for item in resp.json():
            name = item["name"]
            if item["type"] == "dir":
                sub = fetch_dir(f"{path}/{name}" if path else name)
                result[name] = sub
            elif item["type"] == "file" and path:
                # Only include files inside folders (skip root-level README, etc.)
                file_path = f"{path}/{name}" if path else name
                file_resp = requests.get(f"{api_base}/{file_path}", headers=headers)
                file_resp.raise_for_status()
                file_data = file_resp.json()
                if "content" in file_data:
                    content = base64.b64decode(file_data["content"]).decode("utf-8", errors="replace")
                else:
                    r = requests.get(file_data["download_url"], headers=headers)
                    r.raise_for_status()
                    content = r.text
                result[name] = content
        
        return result
    
    tree = fetch_dir("")
    # Tree is {config: {file: content}, LRS: {file: content}, ...} - same format as get_dummy_repo_data()
    return tree

def get_dummy_repo_data():

    repo = {
        "LRS": {
            "japan_data_processor.cs": """
public class JapanDataProcessor {
    public List<Data> Process(string regionCode) {

        if (!GlobalConfig.EnabledRegions.Contains(regionCode)) {
            return new List<Data>();
        }

        if (!FeatureFlags.EnableJapan) {
            return new List<Data>();
        }

        if (regionCode != "JP") {
            return new List<Data>();
        }

        return LoadJapanData();
    }
}
""",
            "us_data_processor.cs": """
public class USDataProcessor {
    public List<Data> Process(string regionCode) {

        if (!GlobalConfig.EnabledRegions.Contains(regionCode)) {
            return new List<Data>();
        }

        if (regionCode != "US") {
            return new List<Data>();
        }

        return LoadUSData();
    }
}
"""
        },
        "Digital360": {
            "japan_sales.cs": """
public class JapanSales {
    public int GetSales(string country) {

        if (!GlobalConfig.EnabledRegions.Contains(country)) {
            return 0;
        }

        if (country == "JP") {
            return 100;
        }

        return 0;
    }
}
"""
        },
        "config": {
            "global_config.cs": """
public static class GlobalConfig {
    public static List<string> EnabledRegions = 
        new List<string> { "US", "EU" };
}
""",
            "feature_flags.cs": """
public static class FeatureFlags {
    public static bool EnableJapan = false;
}
"""
        }
    }

    return repo


# --------------------------------------------
# 2. Flatten Repo + add Config

from pyspark.sql import Row

def build_dataframe_with_config(spark, repo=None):
    """
    Build DataFrame from repo. Use repo from get_repo_data_from_github() or get_dummy_repo_data().
    """
    repo = repo or get_dummy_repo_data()

    # Combine all config files
    config_files = repo.get("config", {})
    combined_config_code = "\n".join(config_files.values())

    rows = []
    idd = 1

    for business_unit, files in repo.items():

        if business_unit == "config":
            continue

        for filename, code in files.items():

            full_code = combined_config_code + "\n\n" + code

            rows.append(
                Row(
                    chunk_id=idd,
                    business_unit=business_unit,
                    filename=filename,
                    file_path=f"{business_unit}/{filename}",
                    code=full_code
                )
            )
            idd += 1

    df = spark.createDataFrame(rows)
    return df


# --------------------------------------------
# 3. Write to Delta Table

# df = build_dataframe_with_config(spark)

# df.write.format("delta") \
#     .mode("overwrite") \
#     .saveAsTable(TABLE_NAME)

# print(f"Delta table '{TABLE_NAME}' created successfully.")
# display(spark.table(TABLE_NAME))

In [ ]:
import os
os.environ["GITHUB_ACCESS_TOKEN"] = ""

### Option A: Use GitHub API (requires GITHUB_ACCESS_TOKEN in .env)

```python
# Fetch from GitHub repo - same format as get_dummy_repo_data()
repo = get_repo_data_from_github("https://github.com/Sourav692/repo_dummy")
df = build_dataframe_with_config(spark, repo=repo)
```

### Option B: Use dummy data (no token needed)

```python
df = build_dataframe_with_config(spark)  # uses get_dummy_repo_data()
```

In [ ]:
# Optional: Use GitHub instead of dummy data (uncomment to run)
# repo = get_repo_data_from_github("https://github.com/Sourav692/repo_dummy")
# df = build_dataframe_with_config(spark, repo=repo)
# df.write.format("delta").mode("overwrite").saveAsTable(TABLE_NAME)

In [12]:
# Fetch from GitHub (requires GITHUB_ACCESS_TOKEN in .env)
repo = get_repo_data_from_github("https://github.com/Sourav692/repo_dummy")
df = build_dataframe_with_config(spark, repo=repo)

In [13]:
display(df)

chunk_id,business_unit,filename,file_path,code
1,Digital360,japan_sales.cs,Digital360/japan_sales.cs,"public static class FeatureFlags { public static bool EnableJapan = false; } public static class GlobalConfig { public static List EnabledRegions = new List { ""US"", ""EU"" }; } public class JapanSales { public int GetSales(string country) { if (!GlobalConfig.EnabledRegions.Contains(country)) { return 0; } if (country == ""JP"") { return 100; } return 0; } }"
2,LRS,japan_data_processor.cs,LRS/japan_data_processor.cs,"public static class FeatureFlags { public static bool EnableJapan = false; } public static class GlobalConfig { public static List EnabledRegions = new List { ""US"", ""EU"" }; } public class JapanDataProcessor { public List Process(string regionCode) { if (!GlobalConfig.EnabledRegions.Contains(regionCode)) { return new List(); } if (!FeatureFlags.EnableJapan) { return new List(); } if (regionCode != ""JP"") { return new List(); } return LoadJapanData(); } }"
3,LRS,us_data_processor.cs,LRS/us_data_processor.cs,"public static class FeatureFlags { public static bool EnableJapan = false; } public static class GlobalConfig { public static List EnabledRegions = new List { ""US"", ""EU"" }; } public class USDataProcessor { public List Process(string regionCode) { if (!GlobalConfig.EnabledRegions.Contains(regionCode)) { return new List(); } if (regionCode != ""US"") { return new List(); } return LoadUSData(); } }"


In [ ]:
TABLE_NAME

## Sync to a vector index -- > This is purely optional - Kept it for future scaling. 

In [ ]:
# Enable Change Data Feed (required for Delta Sync Index)
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("Change Data Feed enabled on the Delta table.")

In [ ]:
from databricks.vector_search.client import VectorSearchClient

ENDPOINT_NAME = "one-env-shared-endpoint-1"

vs_client = VectorSearchClient()

# Create the Vector Search endpoint (skip if it already exists)
try:
    vs_client.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    print(f"Vector Search endpoint '{ENDPOINT_NAME}' created. It may take a few minutes to provision.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{ENDPOINT_NAME}' already exists. Skipping creation.")
    else:
        raise e

In [ ]:
INDEX_NAME = f"{CATALOG}.{SCHEMA}.git_index_raw_index"

# Create the Delta Sync Index with managed embeddings
try:
    # Try to delete the index first if it exists on a different endpoint
    try:
        vs_client.delete_index(index_name=INDEX_NAME)
        print(f"Deleted existing index '{INDEX_NAME}' to recreate on correct endpoint.")
    except Exception:
        pass  # Index doesn't exist or already deleted
    
    index = vs_client.create_delta_sync_index(
        endpoint_name=ENDPOINT_NAME,
        source_table_name=TABLE_NAME,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="filename",
        embedding_model_endpoint_name="databricks-gte-large-en"
    )
    print(f"Vector Search index '{INDEX_NAME}' created successfully.")
    print("Note: Embedding computation may take a few minutes. Wait for the index to become ONLINE before querying.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Index '{INDEX_NAME}' already exists. Skipping creation.")
    else:
        raise e

In [ ]:
import time

# Wait for the index to become ONLINE
vs_index = vs_client.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

status = vs_index.describe()
print(f"Index status: {status.get('status', {})}")

# Poll until the index is ready (optional - you can also check manually in the Databricks UI)
while status.get("status", {}).get("ready") != True:
    print("Index is not ready yet. Waiting 30 seconds...")
    time.sleep(30)
    status = vs_index.describe()
    print(f"Index status: {status.get('status', {})}")

print("Index is ONLINE and ready for queries!")